<a href="https://colab.research.google.com/github/RakyKXD/WithList/blob/main/InvokeAI-Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **InvokeAI - Barren Wardo**
> Note: Probably will only work on Paid Colab.

### How to start?
1.   Run Setup Cell
2.   Run Launcher

Enjoy! ❤️

---

## **Setup**
> Note :
> 1. When prompted, click "Restart".
> 2. Remove # in the 2nd last line to use beta version of InvokeAI.

In [ ]:
# Create directory and install required system dependencies
!mkdir -p /content/invokeai
!sudo apt update -y && sudo apt install -y python3.10 python3.10-venv python3.10-dev libglib2.0-0 libgl1-mesa-glx build-essential python3-opencv libopencv-dev wget curl

# Download and install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Create Python 3.10 virtual environment
!python3.10 -m venv /content/invokeai_venv

# Install InvokeAI directly into the virtual environment
!/content/invokeai_venv/bin/pip install --upgrade pip wheel setuptools
!/content/invokeai_venv/bin/pip install "InvokeAI[xformers]" --use-pep517 --extra-index-url https://download.pytorch.org/whl/cu121

## **Launch InvokeAI**

### **Cloudflare**

In [ ]:
import subprocess
import threading
import time
import socket

def tunnel_thread(port=9090):
    while True:
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break

    print("\n[✓] Servidor InvokeAI detectado en el puerto 9090. Iniciando Cloudflare Tunnel...\n")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    for line in p.stderr:
        if "trycloudflare.com" in line:
            url_start = line.find("https://")
            if url_start != -1:
                print(f"\n==================================================")
                print(f"URL pública de InvokeAI: {line[url_start:].strip()}")
                print(f"==================================================\n")
                break

# Iniciar el hilo del túnel en segundo plano
threading.Thread(target=tunnel_thread, daemon=True, args=(9090,)).start()

# Lanzar InvokeAI sin los argumentos --host ni --port
!/content/invokeai_venv/bin/invokeai-web --root /content/invokeai

In [ ]:
import os

root_dir = "/content/invokeai"
os.makedirs(root_dir, exist_ok=True)

config_content = """# InvokeAI Configuration
host: 0.0.0.0
port: 9090
"""

with open(f"{root_dir}/invokeai.yaml", "w") as f:
    f.write(config_content)

print("[✓] invokeai.yaml creado correctamente en", root_dir)